In [3]:
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

import pandas as pd
import re

from urllib.parse import quote
from pathlib import Path


# ============================================================
# FIND PROJECT ROOT
# ============================================================

def find_project_root():

    current = Path.cwd()

    if current.name == "supermarket_project":
        return current

    if (current / "supermarket_project").exists():
        return current / "supermarket_project"

    for parent in current.parents:

        if parent.name == "supermarket_project":
            return parent

    raise FileNotFoundError(
        "Could not find supermarket_project directory."
    )


BASE_DIR = find_project_root()

RAW_DIR = BASE_DIR / "data" / "raw"

RAW_DIR.mkdir(
    parents=True,
    exist_ok=True
)

print("Current directory:", Path.cwd())
print("Project directory:", BASE_DIR)
print("Raw data directory:", RAW_DIR)


# ============================================================
# FILTER SEARCH RESULTS
# ============================================================

def filter_search_results(df, search_term):

    if df.empty:
        return df

    words = search_term.lower().split()

    filtered_df = df.copy()

    for word in words:

        filtered_df = filtered_df[
            filtered_df["name"]
            .str.lower()
            .str.contains(
                word,
                na=False,
                regex=False
            )
        ]

    return filtered_df.copy()


# ============================================================
# EXTRACT PRODUCTS FROM CURRENT PAGE
# ============================================================

def extract_carrefour_products(driver, search_term):

    wait = WebDriverWait(
        driver,
        20
    )

    # --------------------------------------------------------
    # WAIT FOR PRODUCT CARDS
    # --------------------------------------------------------

    wait.until(
        EC.presence_of_element_located(
            (
                By.CSS_SELECTOR,
                "li.product-card-list__item"
            )
        )
    )

    # --------------------------------------------------------
    # FIND PRODUCT CARDS
    # --------------------------------------------------------

    products = driver.find_elements(
        By.CSS_SELECTOR,
        "li.product-card-list__item"
    )

    print(
        f"Product cards found: {len(products)}"
    )

    products_data = []

    skipped = 0

    # --------------------------------------------------------
    # EXTRACT PRODUCTS
    # --------------------------------------------------------

    for product in products:

        # ----------------------------------------------------
        # PRODUCT NAME
        # ----------------------------------------------------

        name_elements = product.find_elements(
            By.CSS_SELECTOR,
            ".product-card__title-link"
        )

        if not name_elements:

            skipped += 1
            continue

        name_element = name_elements[0]

        name = (
            name_element
            .text
            .strip()
        )

        if not name:

            skipped += 1
            continue

        # ----------------------------------------------------
        # URL
        # ----------------------------------------------------

        product_url = (
            name_element
            .get_attribute("href")
        )

        # ----------------------------------------------------
        # PRICE
        # ----------------------------------------------------

        price_elements = product.find_elements(
            By.CSS_SELECTOR,
            ".product-card__price"
        )

        if not price_elements:

            print(
                f"Skipping product without price: {name}"
            )

            skipped += 1
            continue

        price = (
            price_elements[0]
            .text
            .strip()
        )

        # ----------------------------------------------------
        # BRAND
        # ----------------------------------------------------

        parent_elements = product.find_elements(
            By.CSS_SELECTOR,
            ".product-card__parent"
        )

        if parent_elements:

            brand = (
                parent_elements[0]
                .get_attribute("brand")
            )

        else:

            brand = None

        # ----------------------------------------------------
        # PRODUCT ID
        # ----------------------------------------------------

        if product_url:

            match = re.search(
                r"/(R-[^/]+)/p",
                product_url
            )

            if match:

                product_id = (
                    match.group(1)
                )

            else:

                product_id = None

        else:

            product_id = None

        # ----------------------------------------------------
        # PRICE PER UNIT
        # ----------------------------------------------------

        unit_elements = product.find_elements(
            By.CSS_SELECTOR,
            ".product-card__price-per-unit"
        )

        if unit_elements:

            price_per_unit_raw = (
                unit_elements[0]
                .text
                .strip()
            )

        else:

            price_per_unit_raw = None

        # ----------------------------------------------------
        # STORE PRODUCT
        # ----------------------------------------------------

        products_data.append(
            {
                "searched_item": search_term,
                "supermarket": "Carrefour",
                "product_id": product_id,
                "brand": brand,
                "name": name,
                "price": price,
                "price_per_unit_raw": price_per_unit_raw,
                "url": product_url
            }
        )

    print(
        f"Valid products extracted: "
        f"{len(products_data)}"
    )

    print(
        f"Cards skipped: "
        f"{skipped}"
    )

    # --------------------------------------------------------
    # CREATE DATAFRAME
    # --------------------------------------------------------

    df = pd.DataFrame(
        products_data
    )

    if df.empty:
        return df

    # --------------------------------------------------------
    # CLEAN PRICE
    #
    # "0,84 €" -> 0.84
    # --------------------------------------------------------

    df["price"] = (
        df["price"]
        .str.replace(
            "€",
            "",
            regex=False
        )
        .str.replace(
            ",",
            ".",
            regex=False
        )
        .str.strip()
    )

    df["price"] = pd.to_numeric(
        df["price"],
        errors="coerce"
    )

    # --------------------------------------------------------
    # EXTRACT UNIT
    #
    # "0,84 €/l" -> L
    # "2,50 €/kg" -> KG
    # --------------------------------------------------------

    df["unit"] = (
        df["price_per_unit_raw"]
        .str.extract(
            r"€/([A-Za-zÁÉÍÓÚáéíóú\.]+)"
        )[0]
        .str.replace(
            ".",
            "",
            regex=False
        )
        .str.upper()
    )

    # --------------------------------------------------------
    # CLEAN PRICE PER UNIT
    #
    # "0,84 €/l" -> 0.84
    # --------------------------------------------------------

    df["price_per_unit"] = (
        df["price_per_unit_raw"]
        .str.extract(
            r"([\d,]+)"
        )[0]
        .str.replace(
            ",",
            ".",
            regex=False
        )
    )

    df["price_per_unit"] = pd.to_numeric(
        df["price_per_unit"],
        errors="coerce"
    )

    # --------------------------------------------------------
    # REMOVE RAW COLUMN
    # --------------------------------------------------------

    df = df.drop(
        columns=[
            "price_per_unit_raw"
        ]
    )

    return df


# ============================================================
# SHOPPING LIST
# ============================================================

shopping_list = [
    "leche semidesnatada",
    "manzana",
    "huevos"
]


# ============================================================
# CHROME CONFIGURATION
# ============================================================

options = Options()

options.add_argument(
    "--window-size=1920,1080"
)

driver = webdriver.Chrome(
    options=options
)

wait = WebDriverWait(
    driver,
    20
)


# ============================================================
# OPEN CARREFOUR HOME PAGE
# ============================================================

home_url = "https://www.carrefour.es/"

driver.get(
    home_url
)


# ============================================================
# ACCEPT COOKIES FIRST
# ============================================================

try:

    accept_cookies = wait.until(
        EC.element_to_be_clickable(
            (
                By.XPATH,
                "//button[contains(., 'Aceptar')]"
            )
        )
    )

    accept_cookies.click()

    print("Cookies accepted.")

except:

    print(
        "Cookie button not found or already accepted."
    )


# ============================================================
# SEARCH ALL ITEMS
# ============================================================

all_results = []


try:

    for item in shopping_list:

        print("\n" + "=" * 60)
        print(f"SHOPPING ITEM: {item}")
        print("=" * 60)

        # ----------------------------------------------------
        # BUILD SEARCH URL
        # ----------------------------------------------------

        encoded_term = quote(
            item
        )

        search_url = (
            f"https://www.carrefour.es/"
            f"?query={encoded_term}"
        )

        print(
            f"Searching Carrefour for: {item}"
        )

        print(
            f"URL: {search_url}"
        )

        # ----------------------------------------------------
        # OPEN SEARCH PAGE
        # ----------------------------------------------------

        driver.get(
            search_url
        )

        # ----------------------------------------------------
        # EXTRACT PRODUCTS
        # ----------------------------------------------------

        df_item = extract_carrefour_products(
            driver,
            item
        )

        # ----------------------------------------------------
        # FILTER SEARCH RESULTS
        # ----------------------------------------------------

        df_item = filter_search_results(
            df_item,
            item
        )

        print(
            f"Relevant products after filtering "
            f"for '{item}': "
            f"{len(df_item)}"
        )

        if not df_item.empty:

            all_results.append(
                df_item
            )

finally:

    driver.quit()


# ============================================================
# COMBINE ALL RESULTS
# ============================================================

if all_results:

    df_carrefour = pd.concat(
        all_results,
        ignore_index=True
    )

else:

    df_carrefour = pd.DataFrame()


# ============================================================
# SHOW FINAL RESULTS
# ============================================================

print(
    "\n\nFINAL CARREFOUR SHOPPING LIST RESULTS:"
)

print(
    df_carrefour
)


# ============================================================
# SAVE RESULTS
# ============================================================

if not df_carrefour.empty:

    output_file = (
        RAW_DIR /
        "carrefour_shopping_list.csv"
    )

    df_carrefour.to_csv(
        output_file,
        index=False,
        encoding="utf-8-sig"
    )

    print(
        f"\nDataset saved successfully in:\n"
        f"{output_file}"
    )

Current directory: c:\Users\Giuliana Galassi\OneDrive\Documentos\uc3m\4TO\WEB ANALYTICS\supermarket_project\scrapers
Project directory: c:\Users\Giuliana Galassi\OneDrive\Documentos\uc3m\4TO\WEB ANALYTICS\supermarket_project
Raw data directory: c:\Users\Giuliana Galassi\OneDrive\Documentos\uc3m\4TO\WEB ANALYTICS\supermarket_project\data\raw
Cookies accepted.

SHOPPING ITEM: leche semidesnatada
Searching Carrefour for: leche semidesnatada
URL: https://www.carrefour.es/?query=leche%20semidesnatada


TimeoutException: Message: 
Stacktrace:
	chromedriver!GetHandleVerifier [0x7ff784fd1035+5a95]
	chromedriver!(No symbol) [0x7ff784ef2c70]
	chromedriver!(No symbol) [0x7ff784d14a7d]
	chromedriver!(No symbol) [0x7ff784d6ff91]
	chromedriver!(No symbol) [0x7ff784d7025c]
	chromedriver!(No symbol) [0x7ff784dc2f67]
	chromedriver!(No symbol) [0x7ff784dbfbea]
	chromedriver!(No symbol) [0x7ff784d61d02]
	chromedriver!(No symbol) [0x7ff784d62b13]
	chromedriver!GetHandleVerifier [0x7ff7853f1411+425e71]
	chromedriver!GetHandleVerifier [0x7ff78541ca47+4514a7]
	chromedriver!GetHandleVerifier [0x7ff78541094e+4453ae]
	chromedriver!GetHandleVerifier [0x7ff7850cbf6e+1009ce]
	chromedriver!(No symbol) [0x7ff784f00a05]
	chromedriver!(No symbol) [0x7ff784efbcf4]
	chromedriver!(No symbol) [0x7ff784efbe84]
	chromedriver!(No symbol) [0x7ff784ee66ac]
	KERNEL32!BaseThreadInitThunk [0x7ffe748acd87+17]
	ntdll!RtlUserThreadStart [0x7ffe7626caec+2c]


In [4]:
import requests
import json


url = "https://www.carrefour.es/search-api/query/v1/search"

params = {
    "internal": "true",
    "instance": "x-carrefour",
    "scope": "mobile",
    "lang": "es",
    "session": "empathy",
    "citrusCatalog": "food",
    "enabled": "true",
    "store": "005290",
    "catalog": "food",
    "query": "manzana",
    "page": 1,
    "agroup": "true"
}

headers = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/153.0.0.0 Safari/537.36"
    )
}


response = requests.get(
    url,
    params=params,
    headers=headers,
    timeout=20
)


print("Status code:", response.status_code)
print("Content type:", response.headers.get("Content-Type"))
print("Response length:", len(response.text))


if response.status_code == 200:

    data = response.json()

    print("\nTOP LEVEL KEYS:")
    print(data.keys())

    print("\nFIRST PART OF RESPONSE:")
    print(
        json.dumps(
            data,
            indent=2,
            ensure_ascii=False
        )[:5000]
    )

else:

    print("\nRESPONSE:")
    print(response.text[:1000])

Status code: 403
Content type: text/html; charset=UTF-8
Response length: 6623

RESPONSE:
<!DOCTYPE html><html lang="en-US"><head><title>Just a moment...</title><meta http-equiv="Content-Type" content="text/html; charset=UTF-8"><meta http-equiv="X-UA-Compatible" content="IE=Edge"><meta name="robots" content="noindex,nofollow"><meta name="viewport" content="width=device-width,initial-scale=1"><meta http-equiv="content-security-policy" content="default-src &#39;none&#39;; script-src &#39;nonce-tcF6hHjZzzgowAhvWPpoCJ&#39; &#39;unsafe-eval&#39; https://challenges.cloudflare.com; script-src-attr &#39;none&#39;; style-src &#39;unsafe-inline&#39;; img-src &#39;self&#39; https://challenges.cloudflare.com; connect-src &#39;self&#39; https://challenges.cloudflare.com; frame-src &#39;self&#39; https://challenges.cloudflare.com blob:; child-src &#39;self&#39; https://challenges.cloudflare.com blob:; worker-src blob:; form-action http: https:; base-uri &#39;self&#39;"><style>*{box-sizing:border-box;